# Model Selection and Cross-Validation

## Introduction to Model Selection

In predictive modeling, one of the most critical decisions is selecting the right model for your data. Model selection involves choosing the best model from a set of candidate models based on their performance on unseen data. This process is essential because:

1. **Different models have different strengths and weaknesses**: Some models may perform better on certain types of data or problems.
2. **Models can vary in complexity**: More complex models can capture intricate patterns but risk overfitting.
3. **The goal is generalization**: We want models that perform well on new, unseen data, not just the training data.

### The Two Cultures in Statistical Modeling

As described by Leo Breiman in his influential paper "Statistical Modeling: The Two Cultures," there are two main approaches to statistical modeling:

1. **Data Modeling Culture**: This approach assumes that the data are generated by a given stochastic data model. It focuses on parameter estimation and hypothesis testing. Examples include linear regression, logistic regression, and the Cox model. This culture represents about 98% of all statisticians according to Breiman.

2. **Algorithmic Modeling Culture**: This approach treats the data mechanism as unknown and complex. It uses algorithmic models to find a function that operates on X to predict the responses Y. Examples include decision trees, neural networks, and support vector machines. This culture represents about 2% of statisticians, but many in other fields.

The key difference is in model validation:
- Data modeling culture validates models using goodness-of-fit tests and residual examination
- Algorithmic modeling culture validates models by measuring predictive accuracy

Breiman argues that the focus on data models has led to irrelevant theory, questionable conclusions, and has kept statisticians from working on a large range of interesting current problems.

### The Bias-Variance Tradeoff

At the heart of model selection is the bias-variance tradeoff:

- **Bias**: The error introduced by approximating a real-world problem with a simplified model. High bias can lead to underfitting, where the model fails to capture the underlying pattern in the data.

- **Variance**: The error introduced by the model's sensitivity to fluctuations in the training data. High variance can lead to overfitting, where the model captures noise in the training data rather than the underlying pattern.

The goal is to find a model with the right balance between bias and variance - one that is complex enough to capture the true patterns in the data but simple enough to generalize well to new data.

![Bias-Variance Tradeoff](https://scikit-learn.org/stable/_images/grid_search_cross_validation.png)

*Image source: scikit-learn documentation*

In [1]:
import numpy as np
import pandas as pd
from time import time
from sklearn.model_selection import train_test_split, LeaveOneOut, KFold, StratifiedKFold
from sklearn.linear_model import Lasso, Ridge
from sklearn.datasets import load_iris

# For notebook environment only
try:
    # Only import visualization libraries if we're actually going to use them
    get_ipython().run_line_magic('matplotlib', 'inline')
except (NameError, AttributeError):
    pass

In [ ]:
# Import preprocessing module
try:
    # Try to use IPython magic if in notebook environment
    get_ipython().run_line_magic('run', 'src/preprocessing.py')
except (NameError, AttributeError):
    # If not in notebook environment, import directly
    import sys
    import os
    sys.path.append(os.path.join(os.path.dirname(__file__), 'src'))
    # Import specific variables instead of using star import
    from preprocessing import (
        dataset_1, dataset_2, dataset_3, dataset_4,
        target_1, target_2, target_3, target_4
    )

In [ ]:
# Check the shapes of our four preprocessed datasets
dataset_1.shape, dataset_2.shape, dataset_3.shape, dataset_4.shape

## Model Selection: Cross-Validation

In the next phase of this project we move into developing our machine learning models. We have previously explored data preprocessing, feature engineering, and principal component analysis. Now we need to select the best model for predicting house prices in Ames, Iowa.

To do this effectively, we'll examine three key concepts in model assessment and selection:

1. Using cross-validation to study model variance and estimate performance on unseen data
2. Applying regularization to help our models generalize by preventing overfitting
3. Using ensemble methods to improve prediction accuracy by combining multiple models

### The Purpose of Cross-Validation

Cross-validation is a resampling technique that helps us estimate how well our model will perform on unseen data. It's important to note that cross-validation itself doesn't improve model generalization - it's a diagnostic tool that helps us:

- Estimate model performance more reliably than a single train-test split
- Compare different models or hyperparameter settings
- Detect overfitting by comparing training and validation performance
- Select the model that will likely perform best on new data

## Cross-Validation Techniques

### The Validation Set Approach

The simplest form of cross-validation is the validation set approach, which involves splitting the dataset into two parts:

1. A **training set** used to fit the model
2. A **validation set** (or test set) used to evaluate the model

This approach is straightforward but has limitations:
- The evaluation depends heavily on which data points end up in the training set vs. the validation set
- We don't use all available data for training, which can be problematic with smaller datasets
- We get only a single estimate of test error, which may have high variance

![Train-Test Split](doc/img/Chapter5/5-1.png)

In [4]:
# Imports moved to the top of the file

In [ ]:
# Verify the shapes of our datasets
(dataset_1.shape,
 dataset_2.shape,
 dataset_3.shape,
 dataset_4.shape)

In [6]:
# Verify that the indices of features and targets match
np.testing.assert_allclose(dataset_1.index, target_1.index)
np.testing.assert_allclose(dataset_2.index, target_2.index)
np.testing.assert_allclose(dataset_3.index, target_3.index)
np.testing.assert_allclose(dataset_4.index, target_4.index)

In [7]:
# Create train-test splits for each of our datasets
# We use a 60-40 split and set a random state for reproducibility
ttsplit_1 = train_test_split(dataset_1, target_1, test_size=0.4, random_state=0)
ttsplit_2 = train_test_split(dataset_2, target_2, test_size=0.4, random_state=0)
ttsplit_3 = train_test_split(dataset_3, target_3, test_size=0.4, random_state=0)
ttsplit_4 = train_test_split(dataset_4, target_4, test_size=0.4, random_state=0)

In [ ]:
# Check the shapes of our train-test split components
# The order is: X_train, X_test, y_train, y_test
[obj.shape for obj in ttsplit_1]

In [9]:
# Define a function to fit a model and evaluate its performance
def fit_score(model, data):
    """
    Fit a model on training data and evaluate on test data
    
    Parameters:
    -----------
    model : sklearn estimator
        The model to fit
    data : tuple
        A tuple containing (X_train, X_test, y_train, y_test)
        
    Returns:
    --------
    tuple
        (R² score on test data, training time in seconds)
    """
    X_train = data[0]
    X_test  = data[1]
    y_train = data[2]
    y_test  = data[3]
    
    start = time()
    model.fit(X_train, y_train)
    end = time() - start 
    return model.score(X_test, y_test), end

In [10]:
# Import moved to the top of the file

In [ ]:
# Evaluate Ridge regression on each dataset
print("Ridge Regression Results (R² score, training time):")
print(f"Dataset 1: {fit_score(Ridge(max_iter=1E5), ttsplit_1)}")
print(f"Dataset 2: {fit_score(Ridge(max_iter=1E5), ttsplit_2)}")
print(f"Dataset 3: {fit_score(Ridge(max_iter=1E5), ttsplit_3)}")
print(f"Dataset 4: {fit_score(Ridge(max_iter=1E5), ttsplit_4)}")

In [ ]:
# Evaluate Lasso regression on each dataset
print("Lasso Regression Results (R² score, training time):")
print(f"Dataset 1: {fit_score(Lasso(max_iter=1E4), ttsplit_1)}")
print(f"Dataset 2: {fit_score(Lasso(max_iter=1E5), ttsplit_2)}")
print(f"Dataset 3: {fit_score(Lasso(max_iter=1E4), ttsplit_3)}")
print(f"Dataset 4: {fit_score(Lasso(max_iter=1E5), ttsplit_4)}")

# Note: The results vary across datasets, highlighting the importance of feature engineering
# Dataset 2 (with PCA components) seems to perform well with both Ridge and Lasso

### Leave-One-Out Cross-Validation (LOOCV)

An alternative to the validation set approach is **leave-one-out cross-validation** (LOOCV).

In LOOCV:
1. We set aside one observation as the validation set
2. We train the model on the remaining n-1 observations
3. We compute the prediction error on the held-out observation
4. We repeat this process for each of the n observations
5. We average the n test errors to get the LOOCV estimate

![LOOCV](doc/img/Chapter5/5-3.png)

The LOOCV estimate is given by:

$$\text{CV}_n=\frac{1}{n}\sum_{i=1}^{n}MSE(f_i)$$

Where $f_i$ is the model trained with the ith observation removed.

**Advantages of LOOCV:**
- Uses almost all data for training (n-1 observations)
- Not influenced by random sampling
- Gives the same result every time (deterministic)

**Disadvantages of LOOCV:**
- Computationally expensive (need to fit n models)
- The n training sets are very similar to each other, which can lead to highly correlated model errors

In [68]:
# Import moved to the top of the file

In [70]:
def fit_score_loo(model, dataset, target):
    """
    Perform leave-one-out cross-validation
    
    Parameters:
    -----------
    model : sklearn estimator
        The model to evaluate
    dataset : DataFrame
        The feature dataset
    target : Series
        The target variable
        
    Returns:
    --------
    None
        Prints mean score, variance, and computation time
    """
    loo = LeaveOneOut()
    scores = []
    start = time()
    for train, test in loo.split(dataset, target):
        train = dataset.index[train]
        test = dataset.index[test]

        X_train = dataset.loc[train]
        X_test  = dataset.loc[test]
        y_train = target.loc[train]  # Fixed: was using dataset instead of target
        y_test  = target.loc[test]   # Fixed: was using dataset instead of target
    
        model.fit(X_train, y_train)
        scores.append(model.score(X_test, y_test))

    end = time() - start 
    scores = np.array(scores)
    print("Mean: {:.4f} Variance: {:.4f} Time: {:.2f}s".format(scores.mean(), scores.var(), end))

In [ ]:
# Evaluate Ridge regression using LOOCV
print("Ridge Regression with LOOCV:")
print("Dataset 1:", end=" ")
print(fit_score_loo(Ridge(), dataset_1, target_1))
print("Dataset 2:", end=" ")
print(fit_score_loo(Ridge(), dataset_2, target_2))
print("Dataset 3:", end=" ")
print(fit_score_loo(Ridge(), dataset_3, target_3))
print("Dataset 4:", end=" ")
print(fit_score_loo(Ridge(), dataset_4, target_4))

In [ ]:
# Evaluate Lasso regression using LOOCV
print("Lasso Regression with LOOCV:")
print("Dataset 1:", end=" ")
print(fit_score_loo(Lasso(), dataset_1, target_1))
print("Dataset 2:", end=" ")
print(fit_score_loo(Lasso(), dataset_2, target_2))
print("Dataset 3:", end=" ")
print(fit_score_loo(Lasso(), dataset_3, target_3))
print("Dataset 4:", end=" ")
print(fit_score_loo(Lasso(), dataset_4, target_4))

# Note: LOOCV provides a more robust estimate of model performance
# but is computationally expensive, especially with larger datasets

### K-Fold Cross-Validation

For most practical applications, **k-fold cross-validation** (KCV) offers a good balance between computational efficiency and reliable performance estimation.

In k-fold cross-validation:
1. The dataset is divided into k equally sized folds
2. For each of the k folds:
   - The fold is treated as the validation set
   - The remaining k-1 folds form the training set
   - The model is fit on the training set and evaluated on the validation set
3. The k validation results are averaged to produce a single performance estimate

The k-fold CV estimate is given by:

$$\text{CV}_k=\frac{1}{k}\sum_{i=1}^{k}MSE(f_i)$$

Where $f_i$ is the model trained on all but the ith fold.

![K-Fold CV](doc/img/Chapter5/5-5.png)

**Advantages of k-fold CV:**
- Less computationally expensive than LOOCV
- Provides a good balance between bias and variance in the error estimate
- Typically uses k=5 or k=10, which works well in practice

**Disadvantages of k-fold CV:**
- Results can vary depending on how the data is split into folds
- Uses less training data than LOOCV (though still much more than a simple validation set approach)

In [15]:
# Import moved to the top of the file

In [16]:
# Create a simple example dataset to demonstrate k-fold cross-validation
X_df = pd.DataFrame([
    {'x' : 1},
    {'x' : 2},
    {'x' : 3},
    {'x' : 4},
    {'x' : 5},
    {'x' : 6},
    {'x' : 7},
    {'x' : 8},
    {'x' : 9},
    {'x' : 10},
])

X_df.index = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']

In [17]:
# Remove one row to demonstrate handling with uneven splits
X_df = X_df.drop('E')

In [ ]:
# View the modified dataframe
X_df

In [19]:
# Create a KFold object with 3 splits
kf = KFold(n_splits=3)
splitter = kf.split(X_df)

In [ ]:
# Get the first split to see how the data is divided
train, test = next(splitter)
train, test

In [21]:
# Convert indices to the original dataframe indices
train = X_df.index[train]

In [ ]:
# View the training indices for the first fold
train

In [25]:
def fit_score_kfold(model, dataset, target, folds=5):
    """
    Perform k-fold cross-validation
    
    Parameters:
    -----------
    model : sklearn estimator
        The model to evaluate
    dataset : DataFrame
        The feature dataset
    target : Series
        The target variable
    folds : int, default=5
        Number of folds for cross-validation
        
    Returns:
    --------
    None
        Prints mean score, variance, and computation time
    """
    kf = KFold(n_splits=folds)
    scores = []
    start = time()
    for train, test in kf.split(dataset, target):
        train = dataset.index[train]
        test = dataset.index[test]

        X_train = dataset.loc[train]
        X_test  = dataset.loc[test]
        y_train = target.loc[train]
        y_test  = target.loc[test]
    
        model.fit(X_train, y_train)
        scores.append(model.score(X_test, y_test))
    
    scores = np.array(scores)
    end = time() - start 

    print("Mean: {:.4f} Variance: {:.4f} Time: {:.2f}s".format(scores.mean(), scores.var(), end))

In [ ]:
# Evaluate Ridge regression using 5-fold cross-validation
print("Ridge Regression with 5-fold CV:")
print("Dataset 1:", end=" ")
fit_score_kfold(Ridge(), dataset_1, target_1)
print("Dataset 2:", end=" ")
fit_score_kfold(Ridge(), dataset_2, target_2)
print("Dataset 3:", end=" ")
fit_score_kfold(Ridge(), dataset_3, target_3)
print("Dataset 4:", end=" ")
fit_score_kfold(Ridge(), dataset_4, target_4)

In [ ]:
# Evaluate Lasso regression using 5-fold cross-validation
print("Lasso Regression with 5-fold CV:")
print("Dataset 1:", end=" ")
fit_score_kfold(Lasso(), dataset_1, target_1)
print("Dataset 2:", end=" ")
fit_score_kfold(Lasso(), dataset_2, target_2)
print("Dataset 3:", end=" ")
fit_score_kfold(Lasso(), dataset_3, target_3)
print("Dataset 4:", end=" ")
fit_score_kfold(Lasso(), dataset_4, target_4)

# Note: K-fold CV provides a good balance between computational efficiency
# and reliable performance estimation

### Stratified Cross-Validation

When dealing with classification problems or regression problems with imbalanced target distributions, **stratified cross-validation** ensures that each fold maintains the same distribution of target values as the original dataset.

In [36]:
# Import moved to the top of the file

In [26]:
# Import moved to the top of the file

X, y = load_iris(return_X_y=True)

# First, let's see how regular KFold works
kf = KFold(n_splits=3)
splitter = kf.split(X)

train_index, test_index = next(splitter)

In [ ]:
# View the test indices for the first fold
test_index

In [ ]:
# Check the distribution of target classes in the test set
y[test_index]

In [ ]:
# Check the distribution of target classes in the training set
y[train_index]

In [38]:
# Now let's use StratifiedKFold to maintain class distribution
kf = StratifiedKFold(n_splits=3)
splitter = kf.split(X, y)

train_index, test_index = next(splitter)

In [ ]:
# View the test indices for the first fold with stratification
test_index

In [ ]:
# Check the distribution of target classes in the stratified test set
y[test_index]

In [ ]:
# Check the distribution of target classes in the stratified training set
y[train_index]

# Note: Stratified cross-validation ensures that each fold has approximately
# the same proportion of target classes as the complete dataset

### Bias-Variance Trade-Off for k-Fold Cross-Validation

When choosing between different cross-validation methods, we need to consider the bias-variance trade-off in the error estimate itself:

**LOOCV vs. k-Fold CV:**

- **Bias**: LOOCV has lower bias than k-fold CV when $k < n$. This is because each LOOCV model is trained on $n-1$ observations, which is nearly the entire dataset. In contrast, k-fold CV trains each model on only $(k-1)/k$ of the data, which gives the model less information to learn from.

- **Variance**: LOOCV typically has higher variance than k-fold CV. This is because:
  1. LOOCV involves averaging the performance of $n$ models, whereas k-fold CV averages only $k$ models
  2. The $n$ LOOCV models are highly correlated with each other (each differs by only one observation)
  3. The $k$ k-fold CV models differ from each other by $n/k$ observations, making them less correlated
  4. The mean of highly correlated quantities has higher variance than the mean of less correlated quantities

In practice, k-fold CV with k=5 or k=10 often provides the best balance between bias and variance in the error estimate.

## Introduction to Regularization

Regularization is a technique used to prevent overfitting by adding a penalty term to the loss function that the model optimizes. This penalty discourages complex models by constraining the size of the model parameters.

The two most common regularization techniques for linear models are:

### Ridge Regression (L2 Regularization)

Ridge regression adds the sum of squared coefficients to the loss function:

$$\text{Loss} = \text{MSE} + \alpha \sum_{i=1}^{p} \beta_i^2$$

Where:
- MSE is the mean squared error
- $\alpha$ is the regularization strength
- $\beta_i$ are the model coefficients

Ridge regression shrinks coefficients toward zero but rarely sets them exactly to zero.

### Lasso Regression (L1 Regularization)

Lasso regression adds the sum of absolute coefficient values to the loss function:

$$\text{Loss} = \text{MSE} + \alpha \sum_{i=1}^{p} |\beta_i|$$

Lasso can shrink some coefficients exactly to zero, effectively performing feature selection.

In our cross-validation examples above, we've been using both Ridge and Lasso regression. The next notebook will explore regularization in more detail.

## Summary and Conclusions

In this notebook, we've explored different cross-validation techniques for model selection:

1. **Validation Set Approach**: Simple but limited by data splitting variability
2. **Leave-One-Out Cross-Validation**: Comprehensive but computationally expensive
3. **K-Fold Cross-Validation**: Good balance between reliability and computational cost
4. **Stratified Cross-Validation**: Maintains target distribution across folds

### Key Findings for the Ames Housing Dataset:

1. Both Ridge and Lasso regression models perform well on our preprocessed datasets
2. Dataset 2 (with PCA components) generally shows good performance across models
3. K-fold cross-validation provides reliable performance estimates with reasonable computation time
4. The choice between Ridge and Lasso depends on our goals:
   - Ridge may be better if we want to keep all features but reduce their impact
   - Lasso may be better if we want automatic feature selection

### Lessons from the Two Cultures:

Following Breiman's insights, we can see that:

1. **Predictive accuracy** is a more reliable measure of model quality than goodness-of-fit tests
2. **Algorithmic models** like random forests and support vector machines often provide better predictive accuracy than traditional statistical models
3. **Information extraction** from complex models is possible and can provide more reliable insights than simpler models with poor predictive accuracy
4. **Dimensionality** can be a blessing rather than a curse when using appropriate algorithmic methods

### Next Steps:

1. Explore regularization in more detail to understand how it affects our models
2. Tune hyperparameters (like the regularization strength α) using cross-validation
3. Consider ensemble methods to further improve prediction accuracy
4. Evaluate final models on completely held-out test data
5. Explore algorithmic models like random forests and support vector machines

Cross-validation is a powerful tool for model selection, but it's important to remember that it's just one part of the modeling process. The ultimate goal is to build a model that generalizes well to new, unseen data.